# Notebook 03: Baseline Models

## Overview
This notebook builds the performance floor for the machine learning track. Two simple classifiers are trained: logistic regression and k nearest neighbours. They are easy to understand and fast to train, so they show how much can be achieved with the handcrafted features before stronger models are tried. Every later model is judged against this baseline. Logistic regression uses the full scaled feature set, while k nearest neighbours uses the PCA reduced set because distance based methods work better in fewer dimensions.

## Objectives
1. Train a logistic regression model on the full scaled features.
2. Train a k nearest neighbours model on the PCA features with a tuned value of k.
3. Evaluate both models with accuracy, AUC-ROC, F1, sensitivity, and specificity.
4. Plot the confusion matrix and ROC curves for the stronger baseline.
5. Measure the train versus test accuracy gap and save the better baseline.

In [ ]:
# ----- Environment setup -----
!pip install -q scikit-image seaborn joblib

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ---- Clone the project repository so notebooks can import from src/ ----
REPO       = "lung-colon-cancer-histopathology-ml"
REPO_LOCAL = "/content/" + REPO
REPO_URL   = "https://github.com/hodyek/" + REPO + ".git"

if not os.path.exists(REPO_LOCAL):
    !git clone $REPO_URL $REPO_LOCAL

# Insert at position 0 so our src/ always takes priority over any Colab defaults.
if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

# Quick sanity-check that the import will work before running any cells.
import importlib.util
_spec = importlib.util.find_spec("src.dataset")
if _spec is None:
    raise ImportError(
        "src.dataset not found. Check that the clone succeeded and "
        "that src/__init__.py exists in the repository."
    )
print("src modules found at:", os.path.join(REPO_LOCAL, "src"))

# ---- Project folders on Google Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/" + REPO
DATA_DIR   = os.path.join(DRIVE_ROOT, "data", "lung_colon_image_set")
FIG_DIR    = os.path.join(DRIVE_ROOT, "figures")
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
FEAT_DIR   = os.path.join(DRIVE_ROOT, "features")
for d in (FIG_DIR, MODEL_DIR, FEAT_DIR):
    os.makedirs(d, exist_ok=True)

CLASSES = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
print("Setup complete. CLASSES:", CLASSES)


In [ ]:
# Load the cached features and labels produced in Notebook 02.
import numpy as np, json
y_train = np.load(os.path.join(FEAT_DIR, "y_train.npy"))
y_val   = np.load(os.path.join(FEAT_DIR, "y_val.npy"))
y_test  = np.load(os.path.join(FEAT_DIR, "y_test.npy"))

Xtr_s = np.load(os.path.join(FEAT_DIR, "X_train_scaled.npy"))
Xva_s = np.load(os.path.join(FEAT_DIR, "X_val_scaled.npy"))
Xte_s = np.load(os.path.join(FEAT_DIR, "X_test_scaled.npy"))

Xtr_p = np.load(os.path.join(FEAT_DIR, "X_train_pca.npy"))
Xva_p = np.load(os.path.join(FEAT_DIR, "X_val_pca.npy"))
Xte_p = np.load(os.path.join(FEAT_DIR, "X_test_pca.npy"))

print("Train labels:", y_train.shape, "Test labels:", y_test.shape)

In [ ]:
# Logistic regression on the full scaled features.
from sklearn.linear_model import LogisticRegression
from src.train import fit_and_time
from src.evaluate import get_scores, evaluate_model, print_metrics

logreg = LogisticRegression(max_iter=2000, multi_class="multinomial", n_jobs=-1)
logreg, t_lr = fit_and_time(logreg, Xtr_s, y_train)
lr_pred = logreg.predict(Xte_s)
lr_scores = get_scores(logreg, Xte_s)
lr_metrics = evaluate_model(y_test, lr_pred, lr_scores, CLASSES)
print_metrics("Logistic Regression", lr_metrics)
print(f"  Training time: {t_lr:.1f}s")

In [ ]:
# k nearest neighbours on the PCA features, with a short search over k.
from sklearn.neighbors import KNeighborsClassifier
from src.train import tune, fit_and_time
from src.evaluate import get_scores, evaluate_model, print_metrics

knn_best, knn_params, t_knn_search = tune(
    KNeighborsClassifier(), {"n_neighbors": [3, 5, 7, 9]}, Xtr_p, y_train, cv=3)
print("Best k:", knn_params)
knn_pred = knn_best.predict(Xte_p)
knn_scores = get_scores(knn_best, Xte_p)
knn_metrics = evaluate_model(y_test, knn_pred, knn_scores, CLASSES)
print_metrics("k Nearest Neighbours", knn_metrics)

Both baseline models reach a clearly useful accuracy, which shows the handcrafted features already carry strong signal about the tissue type. Logistic regression and k nearest neighbours usually land close to each other, with one a little ahead depending on the run. These numbers are the floor that the random forest, support vector machine, and gradient boosting models must beat to justify their extra complexity. Reading the AUC and F1 alongside accuracy confirms the result is not driven by a single easy class.

In [ ]:
# Pick the stronger baseline and plot its confusion matrix and ROC curves.
from src.evaluate import plot_confusion_matrix, plot_roc_curves

if lr_metrics["accuracy"] >= knn_metrics["accuracy"]:
    best_name, best_m, best_scores = "Logistic Regression", lr_metrics, lr_scores
else:
    best_name, best_m, best_scores = "k Nearest Neighbours", knn_metrics, knn_scores
print("Stronger baseline:", best_name)

plot_confusion_matrix(best_m["cm"], CLASSES,
    f"{best_name} confusion matrix",
    os.path.join(FIG_DIR, "03_baseline_confusion_matrix.png"))
plot_roc_curves(y_test, best_scores, CLASSES,
    f"{best_name} ROC curves",
    os.path.join(FIG_DIR, "03_baseline_roc.png"))

The confusion matrix shows most images sit on the diagonal, meaning the baseline gets the majority of predictions right. The errors that do appear cluster between the two lung cancer classes, which matches what the sample images suggested earlier. The ROC curves sit well above the diagonal for every class, with the lowest curve usually belonging to lung_aca or lung_scc. This early pattern of lung confusion is the thread to follow through the rest of the project.

In [ ]:
# Overfitting check and save the stronger baseline plus both metric files.
from src.evaluate import overfitting_gap
from src.train import save_model, save_metrics

best_model = logreg if best_name == "Logistic Regression" else knn_best
Xtr_used = Xtr_s if best_name == "Logistic Regression" else Xtr_p
Xte_used = Xte_s if best_name == "Logistic Regression" else Xte_p
tr_acc, te_acc, gap = overfitting_gap(best_model, Xtr_used, y_train, Xte_used, y_test)
print(f"Train accuracy: {tr_acc:.4f}  Test accuracy: {te_acc:.4f}  Gap: {gap:.4f}")

save_model(best_model, os.path.join(MODEL_DIR, "baseline_best.joblib"))
save_metrics(lr_metrics, os.path.join(MODEL_DIR, "logreg_metrics.json"),
             extra={"model": "Logistic Regression", "train_time_s": t_lr,
                    "train_acc": tr_acc if best_name=="Logistic Regression" else None})
save_metrics(knn_metrics, os.path.join(MODEL_DIR, "knn_metrics.json"),
             extra={"model": "k Nearest Neighbours"})
print("Saved baseline model and metrics.")

The gap between training and test accuracy is small for the baseline, which means these simple models are not memorising the training data. A small gap is expected here because logistic regression and k nearest neighbours have limited capacity to overfit. This gives a clean reference point: any stronger model needs to raise accuracy without opening a large train to test gap. The stronger baseline and both metric files are now saved for the final comparison.

## Summary
The baseline models turn the handcrafted features into solid predictions, setting a clear performance floor. Logistic regression and k nearest neighbours both score well, and their errors concentrate between the two lung cancer classes. The train to test gap is small, so there is no sign of overfitting at this level. The next notebooks try a random forest, a support vector machine, and gradient boosting to see how much further the same features can be pushed.